# Aula 01 — laboratório

In [ ]:
import time

inicio = time.time()
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("aula01")
         .master("local[*]")
         .getOrCreate())

print("sessao criada em %.1f s" % (time.time() - inicio))

In [ ]:
sc = spark.sparkContext

print("versao      ", spark.version)
print("master      ", sc.master)
print("app id      ", sc.applicationId)
print("paralelismo ", sc.defaultParallelism)

In [ ]:
dados = [
    ("vendas",  "SP", 1500.0),
    ("vendas",  "RJ", 2300.0),
    ("vendas",  "MG",  900.0),
    ("suporte", "SP",  400.0),
    ("suporte", "RJ",  650.0),
]
df = spark.createDataFrame(dados, schema=["area", "uf", "valor"])

print("linhas:    ", df.count())
print("particoes: ", df.rdd.getNumPartitions())

In [ ]:
inicio = time.time()
filtrado = df.filter(df.valor > 500)
print("filter() 1a chamada em %.4f s  (inclui inicializacao do PySpark)" % (time.time() - inicio))

inicio = time.time()
filtrado = df.filter(df.valor > 500)
print("filter() 2a chamada em %.4f s  (custo real da transformacao)" % (time.time() - inicio))

In [ ]:
inicio = time.time()
total = filtrado.count()
print("count() devolveu %d em %.2f s" % (total, time.time() - inicio))

In [ ]:
filtrado.explain()

In [ ]:
from pyspark.sql import functions as F

(df.groupBy("area")
   .agg(F.sum("valor").alias("total"),
        F.count("*").alias("registros"))
   .orderBy(F.desc("total"))
   .show())

In [ ]:
st = sc.statusTracker()
jobs = sorted(st.getJobIdsForGroup())
print("acoes executadas: 3")
print("jobs disparados:  %d" % len(jobs))
print()

total_tarefas = 0
for jid in jobs:
    info = st.getJobInfo(jid)
    for sid in info.stageIds:
        si = st.getStageInfo(sid)
        if si:
            total_tarefas += si.numTasks
            print("job %-3d stage %-3d %3d tarefas" % (jid, sid, si.numTasks))

print()
print("total de tarefas agendadas: %d" % total_tarefas)

## 10. Encerrar

In [ ]:
spark.stop()
print("sessao encerrada; a UI da 4040 nao responde mais")